# <font color="#418FDE" size="6.5" uppercase>**Density Anomalies**</font>

>Last update: 20260714.
    
By the end of this Lecture, you will be able to:
- Estimate densities using histograms and kernel density estimation. 
- Apply anomaly and novelty detection estimators to identify unusual samples. 
- Use Bernoulli RBMs for simple binary feature representation learning. 


## **1. Density Estimation**

### **1.1. Histogram Density**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_26/Lecture_B/image_01_01.jpg?v=1784031573" width="250">



>* Group values into bins to show distribution
>* Low-density regions may indicate unusual samples

>* Histogram probability depends on bar area
>* Bin width balances detail and smoothness

>* Quickly shows distribution shapes and rare patterns
>* Bin choices limit interpretation, especially in dimensions



In [ ]:
#@title Python Code - Histogram Density

# This example builds a density histogram.
# Bar area represents probability across bins.
# Rare edge values appear in low-density regions.

import numpy as np
import matplotlib.pyplot as plt

# Generate deterministic waiting times in minutes.
rng = np.random.default_rng(42)
typical_waits = rng.gamma(shape=3.0, scale=8.0, size=220)
long_waits = rng.normal(loc=85.0, scale=6.0, size=8)
waiting_times = np.concatenate([typical_waits, long_waits])

# Validate the one-dimensional numeric sample.
if waiting_times.ndim != 1:
    raise ValueError("Expected one column of waiting times.")
if len(waiting_times) == 0:
    raise ValueError("Expected at least one waiting time.")

# Estimate density with equal-width histogram bins.
bin_edges = np.linspace(0.0, 110.0, 23)
density_values, edges = np.histogram(
    waiting_times, bins=bin_edges, density=True
)

# Convert density heights into probability areas.
bin_widths = np.diff(edges)
probability_area = np.sum(density_values * bin_widths)

print("Histogram density example: hospital waiting times.")
print("Total probability area:", round(float(probability_area), 3))
print("Lowest nonempty density:", round(float(density_values[density_values > 0].min()), 4))

# Plot one density histogram for visual interpretation.
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(waiting_times, bins=bin_edges, density=True, color="skyblue", edgecolor="black")
ax.set_title("Histogram Density of Waiting Times")
ax.set_xlabel("Waiting time in minutes")
ax.set_ylabel("Estimated density")
plt.show()



### **1.2. Kernel Density**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_26/Lecture_B/image_01_02.jpg?v=1784031575" width="250">



>* Smooth curves estimate where data values cluster
>* Low-density regions can reveal unusual samples

>* Kernels create local influence around observations
>* Sparse density regions can reveal anomalies

>* Smooths patterns beyond histogram bin artifacts
>* Interpret cautiously using data and context



In [ ]:
#@title Python Code - Kernel Density

# This example estimates a smooth one-dimensional density.
# Kernel density replaces histogram bins with smooth curves.
# Low density regions suggest unusual delivery times.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import KernelDensity
import sklearn

# A fixed generator makes the simulated data reproducible.
rng = np.random.default_rng(42)
normal_times = rng.normal(loc=32, scale=5, size=180)
late_times = rng.normal(loc=55, scale=3, size=20)

# Combine typical and late deliveries into one column.
delivery_times = np.concatenate([normal_times, late_times])
delivery_times = delivery_times.reshape(-1, 1)

# Validate the simple shape expected by KernelDensity.
if delivery_times.shape != (200, 1):
    raise ValueError("Expected 200 delivery times in one column.")

# Fit one kernel density model to the observations.
kde = KernelDensity(kernel="gaussian", bandwidth=3.0)
kde.fit(delivery_times)

# Evaluate density on a smooth grid of possible times.
time_grid = np.linspace(10, 70, 400).reshape(-1, 1)
log_density = kde.score_samples(time_grid)
density = np.exp(log_density)

# Score a few new deliveries for density-based interpretation.
new_times = np.array([[30], [45], [65]])
new_density = np.exp(kde.score_samples(new_times))

print("scikit-learn version:", sklearn.__version__)
print("Density near 30 minutes:", round(float(new_density[0]), 4))
print("Density near 45 minutes:", round(float(new_density[1]), 4))
print("Density near 65 minutes:", round(float(new_density[2]), 4))

# Plot the smooth KDE curve with the observed samples.
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(time_grid.ravel(), density, label="Kernel density estimate")
ax.scatter(delivery_times.ravel(), np.zeros(200), s=12, alpha=0.35, label="Observed deliveries")

ax.set_title("Kernel Density Estimate of Delivery Times")
ax.set_xlabel("Delivery time in minutes")
ax.set_ylabel("Estimated density")
ax.legend()
plt.show()



### **1.3. Choosing Bandwidth**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_26/Lecture_B/image_01_03.jpg?v=1784031577" width="250">



>* Bandwidth controls KDE smoothness and detail
>* Balance noise reduction with preserving real patterns

>* Bandwidth depends on data size and noise
>* Good choices reveal reliable patterns

>* Use defaults, then compare bandwidth choices
>* Check scale and anomaly detection effects



In [ ]:
#@title Python Code - Choosing Bandwidth

# This example compares kernel density bandwidth choices.
# Bandwidth controls smoothness in density estimation.
# The plot shows underfitting and overfitting effects.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import KernelDensity

# A fixed generator makes the sample reproducible.
rng = np.random.default_rng(42)

# Two spending groups create a distribution with structure.
low_spenders = rng.normal(loc=35, scale=6, size=80)
high_spenders = rng.normal(loc=75, scale=8, size=60)
spending = np.concatenate((low_spenders, high_spenders))

# Validation keeps the example safe and understandable.
if spending.ndim != 1 or spending.size < 20:
    raise ValueError("Expected one numeric spending column.")

# KernelDensity expects a two-dimensional feature matrix.
spending_column = spending.reshape(-1, 1)
spending_grid = np.linspace(10, 105, 400).reshape(-1, 1)

# These bandwidths show too little, balanced, and too much smoothing.
bandwidths = [1.0, 5.0, 15.0]
labels = ["bandwidth 1: jagged", "bandwidth 5: balanced", "bandwidth 15: oversmoothed"]

# One figure keeps the comparison focused.
fig, ax = plt.subplots(figsize=(9, 5))

# The histogram gives a simple reference for the observed data.
ax.hist(spending, bins=18, density=True, alpha=0.25, color="gray", label="histogram")

# Each KDE curve uses the same data but a different bandwidth.
for bandwidth, label in zip(bandwidths, labels):
    kde = KernelDensity(kernel="gaussian", bandwidth=bandwidth)
    kde.fit(spending_column)
    density = np.exp(kde.score_samples(spending_grid))
    ax.plot(spending_grid.ravel(), density, linewidth=2, label=label)

# Labels connect the plot to the spending example.
ax.set_title("Choosing Bandwidth for Kernel Density Estimation")
ax.set_xlabel("Daily customer spending in dollars")
ax.set_ylabel("Estimated density")
ax.legend()

plt.show()



## **2. Anomaly Detection**

### **2.1. Novelty Versus Outliers**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_26/Lecture_B/image_02_01.jpg?v=1784031566" width="250">



>* Outliers already exist within training data
>* Novelties appear after learning normal patterns

>* Outlier detection handles contaminated training data
>* Novelty detection needs clean normal examples

>* Unusual samples need review, not automatic judgment
>* Define normal context before flagging anomalies



### **2.2. One Class SVM**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_26/Lecture_B/image_02_02.jpg?v=1784031568" width="250">



>* Learns normal patterns from clean data
>* Flags unusual machine or network behavior

>* Kernels capture complex normal data shapes
>* Careful tuning balances false anomaly errors

>* Use clean, scaled, representative training data
>* Treat flags as prompts for investigation



In [ ]:
#@title Python Code - One Class SVM

# This example trains a One Class SVM.
# It learns normal points without anomaly labels.
# The plot shows normal and unusual predictions.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler

# A fixed generator makes the example repeatable.
rng = np.random.default_rng(42)

# Most training points represent normal behavior.
normal_train = rng.normal(loc=0.0, scale=0.7, size=(160, 2))

# Test data mixes normal points with obvious unusual points.
normal_test = rng.normal(loc=0.0, scale=0.7, size=(60, 2))
anomaly_test = rng.uniform(low=3.0, high=5.0, size=(12, 2))
test_points = np.vstack((normal_test, anomaly_test))

# Validate the simple two-feature shape before modeling.
if normal_train.shape[1] != 2 or test_points.shape[1] != 2:
    raise ValueError("This lesson expects exactly two numeric features.")

# Scaling keeps both features equally important.
scaler = StandardScaler()
scaled_train = scaler.fit_transform(normal_train)
scaled_test = scaler.transform(test_points)

# The model learns a boundary around normal training data.
model = OneClassSVM(kernel="rbf", gamma="scale", nu=0.08)
model.fit(scaled_train)

# Predictions are 1 for normal and -1 for unusual.
predictions = model.predict(scaled_test)
anomaly_count = int(np.sum(predictions == -1))
normal_count = int(np.sum(predictions == 1))

print("scikit-learn version:", sklearn.__version__)
print("Predicted normal test samples:", normal_count)
print("Predicted unusual test samples:", anomaly_count)

# A grid lets us draw the learned normal region.
x_values = np.linspace(-3.0, 5.5, 180)
y_values = np.linspace(-3.0, 5.5, 180)
xx, yy = np.meshgrid(x_values, y_values)
grid_points = np.c_[xx.ravel(), yy.ravel()]

# The decision function is positive inside the boundary.
scaled_grid = scaler.transform(grid_points)
grid_scores = model.decision_function(scaled_grid).reshape(xx.shape)

# One figure shows training data, predictions, and boundary.
fig, ax = plt.subplots(figsize=(7, 5))
ax.contour(xx, yy, grid_scores, levels=[0], colors="black", linewidths=2)
ax.scatter(normal_train[:, 0], normal_train[:, 1], s=18, label="training normal")

# Color test points by the model prediction.
normal_mask = predictions == 1
anomaly_mask = predictions == -1
ax.scatter(test_points[normal_mask, 0], test_points[normal_mask, 1], s=35, label="predicted normal")
ax.scatter(test_points[anomaly_mask, 0], test_points[anomaly_mask, 1], s=45, label="predicted unusual")

ax.set_title("One Class SVM anomaly detection")
ax.set_xlabel("Feature 1")
ax.set_ylabel("Feature 2")
ax.legend()
plt.show()



### **2.3. Isolation Forest**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_26/Lecture_B/image_02_03.jpg?v=1784031569" width="250">



>* Anomalies separate faster through random splits
>* Finds rare cases without modeling normal density

>* Random trees isolate samples through feature splits
>* Shorter paths suggest more unusual observations

>* Scales well across many anomaly detection domains
>* Scores need careful context and thresholds



In [ ]:
#@title Python Code - Isolation Forest

# Demonstrate Isolation Forest on simple two-dimensional data.
# Random splits isolate unusual points more quickly.
# The plot highlights predicted anomalies in red.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.ensemble import IsolationForest

# Create a compact normal cluster and a few unusual points.
rng = np.random.default_rng(42)
normal_points = rng.normal(loc=0.0, scale=0.7, size=(180, 2))

# Add points far from the main cluster.
anomaly_points = np.array([[3.2, 3.0], [3.5, -2.8], [-3.0, 3.1], [-3.4, -2.7]])
X = np.vstack([normal_points, anomaly_points])

# Validate the small teaching dataset before modeling.
if X.shape != (184, 2):
    raise ValueError("Expected 184 rows and 2 columns.")

# Fit one Isolation Forest to learn what looks unusual.
model = IsolationForest(contamination=0.04, random_state=42)
predicted_labels = model.fit_predict(X)

# Convert model labels into a beginner-friendly Boolean mask.
anomaly_mask = predicted_labels == -1
normal_mask = predicted_labels == 1

# Print concise results that connect scores to flagged samples.
print("scikit-learn version:", sklearn.__version__)
print("Samples flagged as anomalies:", int(anomaly_mask.sum()))
print("Lower scores mean more unusual samples.")

# Plot normal and anomalous samples on one clear axes.
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(X[normal_mask, 0], X[normal_mask, 1], label="Predicted normal", alpha=0.75)
ax.scatter(X[anomaly_mask, 0], X[anomaly_mask, 1], label="Predicted anomaly", color="red", s=70)

# Label the feature space used by the model.
ax.set_title("Isolation Forest anomaly detection")
ax.set_xlabel("Feature 1")
ax.set_ylabel("Feature 2")
ax.legend()
plt.show()



## **3. Bernoulli RBM Features**

### **3.1. Local Outlier Factor**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_26/Lecture_B/image_03_01.jpg?v=1784031564" width="250">



>* Finds sparse points using nearby density
>* Checks unusual RBM feature combinations locally

>* Normal samples share nearby density patterns
>* Outliers are locally rare feature combinations

>* RBM features shape LOF anomaly usefulness
>* Interpret scores with neighborhood and domain context



In [ ]:
#@title Python Code - Local Outlier Factor

# This example finds local density anomalies.
# Binary features mimic learned RBM activations.
# The plot highlights locally unusual samples.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import LocalOutlierFactor
import sklearn

# Create two common binary activation patterns.
rng = np.random.default_rng(42)
cluster_a = rng.binomial(1, [0.9, 0.8, 0.1, 0.1], size=(45, 4))
cluster_b = rng.binomial(1, [0.1, 0.2, 0.9, 0.8], size=(45, 4))

# Add rare combinations that mix both patterns.
unusual = np.array([[1, 1, 1, 1], [0, 0, 0, 0], [1, 0, 1, 0]])
features = np.vstack([cluster_a, cluster_b, unusual])
labels = np.array(["common"] * 90 + ["unusual"] * 3)

# Validate the small binary feature matrix.
if features.shape != (93, 4):
    raise ValueError("Expected 93 samples with 4 binary features.")

# Fit Local Outlier Factor on the feature representation.
lof = LocalOutlierFactor(n_neighbors=10, contamination=0.08)
predicted = lof.fit_predict(features)
outlier_score = -lof.negative_outlier_factor_

# Project four binary features into two readable coordinates.
x_axis = features[:, 0] + features[:, 1]
y_axis = features[:, 2] + features[:, 3]

# Print concise results for the most unusual samples.
top_indices = np.argsort(outlier_score)[-5:][::-1]
print("scikit-learn version:", sklearn.__version__)
print("Higher LOF score means more locally unusual.")
print("Top sample indices:", top_indices.tolist())
print("Top LOF scores:", np.round(outlier_score[top_indices], 2).tolist())
print("Known rare samples flagged:", int(np.sum(predicted[labels == "unusual"] == -1)), "of 3")

# Plot common samples and LOF-flagged outliers.
fig, ax = plt.subplots(figsize=(7, 5))
normal_mask = predicted == 1
outlier_mask = predicted == -1

ax.scatter(x_axis[normal_mask], y_axis[normal_mask], c="tab:blue", label="LOF normal")
ax.scatter(x_axis[outlier_mask], y_axis[outlier_mask], c="tab:red", label="LOF outlier")
ax.set_title("Local Outlier Factor on Binary Feature Activations")
ax.set_xlabel("Number of active features in group 1")
ax.set_ylabel("Number of active features in group 2")
ax.legend()
plt.show()



### **3.2. Elliptic Envelope**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_26/Lecture_B/image_03_02.jpg?v=1784031562" width="250">



>* Learns an elliptical normal data region
>* Flags far outside points as anomalies

>* Best for one elliptical normal cluster
>* Flags unusual feature combinations, but has limits

>* RBMs turn binary inputs into compact features
>* Elliptic Envelope flags unusual learned patterns



In [ ]:
#@title Python Code - Elliptic Envelope

# This example detects unusual learned binary features.
# An RBM compresses binary patterns into hidden features.
# Elliptic Envelope flags samples outside the feature cloud.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.neural_network import BernoulliRBM
from sklearn.covariance import EllipticEnvelope

# A fixed generator makes the synthetic binary data repeatable.
rng = np.random.default_rng(42)
normal_profiles = rng.binomial(1, 0.25, size=(180, 12))

# These profiles contain unusual combinations of active binary features.
anomaly_profiles = rng.binomial(1, 0.85, size=(20, 12))
binary_profiles = np.vstack((normal_profiles, anomaly_profiles))

# This check keeps the lesson focused on binary input features.
unique_values = np.unique(binary_profiles)
if not np.array_equal(unique_values, np.array([0, 1])):
    raise ValueError("The feature matrix must contain only zeros and ones.")

# The RBM learns two compact hidden features from binary inputs.
rbm = BernoulliRBM(n_components=2, learning_rate=0.05, n_iter=20, random_state=42)
learned_features = rbm.fit_transform(binary_profiles)

# The envelope learns the main elliptical cloud in RBM feature space.
envelope = EllipticEnvelope(contamination=0.10, random_state=42)
predicted_labels = envelope.fit_predict(learned_features)

# Negative one means the sample is outside the learned envelope.
anomaly_mask = predicted_labels == -1
found_anomalies = int(np.sum(anomaly_mask))

print(f"scikit-learn version: {sklearn.__version__}")
print(f"RBM feature shape: {learned_features.shape[0]} samples by {learned_features.shape[1]} features")
print(f"Elliptic Envelope flagged {found_anomalies} possible anomalies")

# The plot shows anomalies outside the main learned feature cloud.
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(learned_features[~anomaly_mask, 0], learned_features[~anomaly_mask, 1], label="inside envelope", alpha=0.75)
ax.scatter(learned_features[anomaly_mask, 0], learned_features[anomaly_mask, 1], label="flagged anomaly", alpha=0.9)
ax.set_title("Elliptic Envelope on RBM Features")

ax.set_xlabel("RBM hidden feature 1")
ax.set_ylabel("RBM hidden feature 2")
ax.legend()
plt.show()



### **3.3. Bernoulli RBM**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_26/Lecture_B/image_03_03.jpg?v=1784031571" width="250">



>* Learns patterns from binary on-off features
>* Connects visible data to hidden representations

>* Hidden units compress binary inputs into features
>* Learned features support downstream machine learning

>* Learn normal binary patterns to flag anomalies
>* Use meaningful data and domain judgment



In [ ]:
#@title Python Code - Bernoulli RBM

# Learn binary features with a Bernoulli RBM.
# Hidden units summarize recurring pixel patterns.
# The plot shows learned digit-like feature templates.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_digits
from sklearn.neural_network import BernoulliRBM

# Load small packaged digit images for a binary-feature example.
digits = load_digits()
images = digits.images

# Validate the expected image shape before reshaping.
if images.shape[1:] != (8, 8):
    raise ValueError("Expected 8 by 8 digit images.")

# Convert grayscale pixels into on-off Bernoulli features.
flat_pixels = images.reshape(images.shape[0], -1)
binary_pixels = (flat_pixels > 8).astype(float)

# Train one small RBM to learn hidden binary feature detectors.
rbm = BernoulliRBM(
    n_components=16, learning_rate=0.05, n_iter=20, random_state=42
)
rbm.fit(binary_pixels)

# Transform examples into learned hidden feature probabilities.
hidden_features = rbm.transform(binary_pixels[:5])
mean_activation = hidden_features.mean(axis=0)

print("scikit-learn version:", sklearn.__version__)
print("Input binary features per image:", binary_pixels.shape[1])
print("Learned hidden features:", rbm.n_components)
print("Mean activation of first 5 hidden features:", np.round(mean_activation[:5], 2))

# Display the learned visible patterns for all hidden units.
feature_grid = rbm.components_.reshape(4, 4, 8, 8)
feature_grid = feature_grid.transpose(0, 2, 1, 3).reshape(32, 32)

fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(feature_grid, cmap="gray_r")
ax.set_title("Bernoulli RBM learned feature templates")
ax.set_xlabel("Visible pixel position across templates")
ax.set_ylabel("Visible pixel position across templates")
ax.set_xticks([])
ax.set_yticks([])
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Density Anomalies**</font>


In this lecture, you learned to:
- Estimate densities using histograms and kernel density estimation. 
- Apply anomaly and novelty detection estimators to identify unusual samples. 
- Use Bernoulli RBMs for simple binary feature representation learning. 

In the next Module (Module 27), we will go over 'Cross Validation'